# FF3: Fingerprinting und Consent

Setzen Websites Fingerprinting bereits vor einer Entscheidung im Consent-Banner oder trotz einer Ablehnung ein? Hypothese H3: Third-Party-Skripte rufen Fingerprinting-Schnittstellen bereits vor der Entscheidung auf. Auch nach einer Ablehnung finden weiterhin Third-Party-Fingerprinting-Aufrufe statt.

# Vorbereitung

- Setzen der Projektwurzel auf Workspace-Root
- Einlesen der Daten (je Consent-Aktion eine Datei)

In [ ]:
# Projektwurzel = erster Ordner nach oben mit pyproject.toml
# Pfade gelten ab Wurzel
import os
import sys
from pathlib import Path

ROOT = next(ordner for ordner in [Path.cwd(), *Path.cwd().parents] if (ordner / "pyproject.toml").exists())
os.chdir(ROOT)
# Importe aus dem Repo möglich machen
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projektwurzel:", ROOT)

In [ ]:
# Rohdaten einlesen: je Consent-Aktion eine JSONL-Datei, jede Zeile ein JSON-Objekt
# Konfigurationszeilen (eine je Start des Crawlers, erkennbar an path_to_output) getrennt von den Messzeilen halten
import json

ORDNER = Path("results/Forschungsfrage 3")
# Aktion -> Datei. keine_aktion ist die Vergleichsbasis: Banner wird nicht angefasst
DATEIEN = {
    "keine_aktion": ORDNER / "ff3_keine_aktion.jsonl",
    "zustimmen": ORDNER / "ff3_zustimmen.jsonl",
    "ablehnen": ORDNER / "ff3_ablehnen.jsonl",
}
AKTIONEN = list(DATEIEN)
# Kurze Namen für Achsen und Tabellen
KURZ = {"keine_aktion": "keine Aktion", "zustimmen": "Zustimmung", "ablehnen": "Ablehnung"}

configs = {} # Aktion -> Liste der Konfigurationszeilen (eine je Start des Crawlers)
zeilen = {}  # Aktion -> Liste der Messzeilen
for name, pfad in DATEIEN.items():
    configs[name], zeilen[name] = [], []
    with open(pfad, encoding="utf-8") as datei:
        for line in datei:
            if not line.strip():
                continue
            zeile = json.loads(line)
            (configs[name] if "path_to_output" in zeile else zeilen[name]).append(zeile)
    n_websites = len({zeile["website"] for zeile in zeilen[name]})
    print(f"{name}: {len(configs[name])} Konfigurationszeilen, {len(zeilen[name])} Messzeilen, {n_websites} Websites")

# Verarbeitung

- Je Website und Aktion nur die letzte gültige Messung
- Konfiguration prüfen: alle Starts mit denselben Einstellungen, Läufe unterscheiden sich nur in der Consent-Aktion
- Vollständige Websites: in allen drei Läufen gemessen
- Fehlerhafte Websites aus dem Panel nehmen: HTTP-Fehler, Fehler- oder Bot-Seite, Weiterleitung auf fremde Domain, leerer Titel (an beiden Messpunkten)

In [ ]:
# Je (Website, Aktion) nur die letzte gültige Zeile behalten. Mehrere Zeilen entstehen durch Fehler und Wiederholung
# Gültig = success und internet_ok, wie beim Fortsetzen des Crawlers (core/logging.py, done_crawls)
from collections import Counter

def gueltig(zeile):
    return zeile["success"] and zeile["internet_ok"]

def art(zeile):
    if not gueltig(zeile):
        return "fehler"
    if "stats_vor_consent" in zeile:
        return "gemessen"
    return "unbekannt"

letzte = {}
for aktion in AKTIONEN:
    letzte[aktion] = {}
    # Reihenfolge der Datei = Reihenfolge der Messung: spätere Zeile ersetzt frühere
    for zeile in zeilen[aktion]:
        bisher = letzte[aktion].get(zeile["website"])
        # Gültige Zeile ersetzt alles, Fehlerzeile nur eine Fehlerzeile
        if bisher is None or gueltig(zeile) or not gueltig(bisher):
            letzte[aktion][zeile["website"]] = zeile
    print(f"{aktion}: {len(zeilen[aktion])} Zeilen -> {len(letzte[aktion])} Websites", dict(Counter(art(zeile) for zeile in letzte[aktion].values())))

In [ ]:
# Konfiguration prüfen: alle Starts eines Laufs mit denselben Werten, sonst passen die Zeilen nicht zusammen
# Die Läufe dürfen sich nur in consent_action unterscheiden. consent_window: feste Zeit für Bannersuche und Klick,
# damit der zweite Messpunkt in allen Läufen gleich spät kommt
import pandas as pd

FELDER = ["consent_action", "consent_window", "allow_3p", "js_cookie_block", "loading_time", "visit_time_after_consent", "site_timeout", "batch_size", "batch_seed"]
for aktion in AKTIONEN:
    assert all(config[feld] == configs[aktion][-1][feld] for config in configs[aktion] for feld in FELDER), aktion
pd.DataFrame({aktion: {feld: configs[aktion][-1][feld] for feld in FELDER} for aktion in AKTIONEN})

In [ ]:
# Vollständig = in allen drei Läufen gemessen. Nur so vergleicht man dieselbe Website mit verschiedenen Aktionen
websites = sorted(set().union(*(letzte[aktion] for aktion in AKTIONEN)))
vollstaendig = [website for website in websites
                if all(website in letzte[aktion] and art(letzte[aktion][website]) == "gemessen" for aktion in AKTIONEN)]
print(f"{len(websites)} Websites: {len(vollstaendig)} vollständig, {len(websites) - len(vollstaendig)} unvollständig")

In [ ]:
# Messung an beiden Messpunkten prüfen. Ein Problem in einem Lauf nimmt die ganze Website aus dem Panel
# Fehler- oder Bot-Seite am Titel erkennen: Bot-Sperre, HTTP-Fehlerseite, Domain zu verkaufen oder nur reserviert
import re

FEHLERTITEL = re.compile(r"just a moment|captcha|attention required|access denied|forbidden|^404|not found|error page"
                         r"|is for sale|hugedomains|réservé|a bien été créé|nom de domaine déposé", re.IGNORECASE)

def probleme(website, aktion):
    zeile = letzte[aktion][website]
    gefunden = set()
    # HTTP-Fehler der Startseite, z. B. 403 bei einer Bot-Sperre
    if (zeile["stats_vor_consent"].get("status") or 0) >= 400:
        gefunden.add("HTTP-Fehler")
    for punkt in ("stats_vor_consent", "stats_nach_consent"):
        stats = zeile[punkt]
        titel = stats["title"].strip()
        if not titel:
            gefunden.add("leerer Titel")
        elif FEHLERTITEL.search(titel):
            gefunden.add("Fehler- oder Bot-Seite")
        # Weiterleitung auf fremde Domain: deren Skripte und Cookies zählen sonst als 3P
        if stats["cross_domain_redirect"]:
            gefunden.add("Weiterleitung fremde Domain")
    return gefunden

# Website -> Gründe über alle Läufe
fehlerhaft = {}
for website in vollstaendig:
    gruende = set().union(*(probleme(website, aktion) for aktion in AKTIONEN))
    if gruende:
        fehlerhaft[website] = sorted(gruende)
panel = [website for website in vollstaendig if website not in fehlerhaft]
print(f"{len(vollstaendig)} vollständig -> {len(fehlerhaft)} fehlerhaft, {len(panel)} im Panel")
print("Gründe (eine Website kann mehrere haben):", dict(Counter(grund for gruende in fehlerhaft.values() for grund in gruende)))
# Zum Prüfen von Hand: Titel, die als Fehler- oder Bot-Seite gelten
treffer = Counter(letzte[aktion][website][punkt]["title"].strip()[:80] for website in vollstaendig for aktion in AKTIONEN
                  for punkt in ("stats_vor_consent", "stats_nach_consent") if FEHLERTITEL.search(letzte[aktion][website][punkt]["title"]))
for titel, anzahl in treffer.most_common():
    print(f"  {anzahl:2d} x {titel}")

**Panel**
- 1000 Websites, 752 in allen drei Läufen gemessen, 575 ohne Fehler im Panel
- economictimes.com fehlt in zwei Läufen: der Crawler überspringt sie (SKIP_SITES in core/consent_crawl.py), so viele FP-Aufrufe, dass der Treiber hängt. In der Thesis als ausgeschlossen nennen
- Die Thesis beschreibt zwei Läufe (zustimmen, ablehnen), gemessen wurden drei: keine Aktion ist die Vergleichsbasis. Text und Tabelle tab:Messreihen ergänzen

# DataFrames

- snap: eine Zeile je Website und Aktion mit Panel, Reihenfolge, Klick, geprüfter Entscheidung, TCF, Cookies, Titel, Screenshot
- Wertung der Methoden nach Tabelle tab:Schnittstellen (live: devicePixelRatio zählt mit, Storage zählt nicht als FP)
- calls: Aufrufe je Aktion, Messpunkt, Methode, Partei und Quelle

In [ ]:
# snap: eine Zeile je Website und Aktion. Für Wirksamkeit der Aktion, Cookies, Reihenfolge und Screenshot-Sichtung
# Die Aufrufe stehen in calls
PUNKTE = {"vor": "stats_vor_consent", "nach": "stats_nach_consent"}
panel_menge = set(panel)

snap_zeilen = []
for aktion in AKTIONEN:
    for website, zeile in letzte[aktion].items():
        eintrag = {"aktion": aktion, "website": website, "art": art(zeile), "im_panel": website in panel_menge}
        if eintrag["art"] == "gemessen":
            nach = zeile["stats_nach_consent"] or {}
            check = nach.get("consent_check") or {}
            tcf = check.get("tcf") or {}
            eintrag |= {
                "crawl_time": zeile["crawl_time"],
                # Klick: Button gefunden und geklickt (keine Aktion: nie)
                "consent_klick": bool(nach.get("consent_ok")),
                # Gespeicherte Entscheidung laut TCF, OneTrust oder Cookiebot: accept, reject oder None (keine bzw. nicht lesbar)
                "entscheidung": check.get("consent_decision"),
                # Passt die Entscheidung zur Aktion? True / False / None (nicht prüfbar). Keine Aktion: True = Seite hat nichts selbst entschieden
                "geprueft": check.get("consent_verified"),
                "tcf_status": tcf.get("status"),
                "tcf_vendors": tcf.get("n_vendors"),
                "klick_stufe": check.get("consent_stage"),
                "button": (check.get("consent_click") or {}).get("text"),
            }
            for punkt, key in PUNKTE.items():
                stats = zeile[key] or {}
                eintrag |= {
                    # Domains mit unpartitionierten 3P-Cookies, als Menge für "neu nach der Entscheidung"
                    f"cookie_domains_3p_{punkt}": frozenset(stats.get("cookie_sources_3p", [])),
                    f"titel_{punkt}": stats.get("title"),
                    f"screenshot_{punkt}": stats.get("screenshot"),
                }
        snap_zeilen.append(eintrag)

snap = pd.DataFrame(snap_zeilen)
snap["crawl_time"] = pd.to_datetime(snap["crawl_time"])
# Reihenfolge der drei Besuche einer Website (1 = zuerst). Wechselt reihum, damit sich Reihenfolgeeffekte ausgleichen
snap["position"] = snap.groupby("website")["crawl_time"].rank(method="first")
print(len(snap), "Zeilen")
pd.crosstab(snap["aktion"], snap["art"]).loc[AKTIONEN]

In [ ]:
import pandas as pd

# Wertung der Methoden wie Tabelle tab:Schnittstellen der Thesis. Schlüssel "api.method" wie in events, nicht nur api:
# getBattery hat api "Navigator", gehört aber zu Gerät. getImageData (stark) und measureText (schwach) haben beide api "canvas2D"
METHODEN = pd.DataFrame([
    ("canvas.toDataURL", "Canvas", "stark"),
    ("canvas.toBlob", "Canvas", "stark"),
    ("canvas2D.getImageData", "Canvas", "stark"),
    ("canvas2D.measureText", "Canvas", "schwach"),
    ("canvas2D.isPointInPath", "Canvas", "schwach"),
    ("WebGL.getParameter", "WebGL", "stark"),
    ("WebGL.getSupportedExtensions", "WebGL", "stark"),
    ("WebGL.readPixels", "WebGL", "stark"),
    ("audio.getChannelData", "Audio", "stark"),
    ("webaudio.OfflineAudioContext", "Audio", "stark"),
    ("webaudio.AudioContext", "Audio", "schwach"),
    ("Navigator.userAgent", "Browser", "schwach"),
    ("Navigator.userAgentData", "Browser", "schwach"),
    ("NavigatorUAData.getHighEntropyValues", "Browser", "schwach"),
    ("Navigator.platform", "Browser", "schwach"),
    ("Navigator.plugins", "Browser", "schwach"),
    ("Navigator.language", "Browser", "schwach"),
    ("Navigator.languages", "Browser", "schwach"),
    ("Navigator.cookieEnabled", "Browser", "schwach"),
    ("Navigator.doNotTrack", "Browser", "schwach"),
    ("DateTimeFormat.resolvedOptions", "Browser", "schwach"),
    ("Screen.width", "Bildschirm", "schwach"),
    ("Screen.height", "Bildschirm", "schwach"),
    ("Screen.colorDepth", "Bildschirm", "schwach"),
    ("Window.devicePixelRatio", "Bildschirm", "schwach"),
    ("Navigator.getBattery", "Gerät", "schwach"),
    ("MediaDevices.enumerateDevices", "Gerät", "schwach"),
    ("SpeechSynthesis.getVoices", "Gerät", "schwach"),
    ("StorageManager.estimate", "Gerät", "schwach"),
    ("local/session storage.getItem", "Storage", "kein FP"),
    ("local/session storage.setItem", "Storage", "kein FP"),
], columns=["methode", "kategorie", "wertung"])
# Live gibt es kein Archiv: devicePixelRatio ist messbar und zählt. Storage ist kein FP, bleibt aber für die Frage nach dem Ausweichen drin
METHODEN["zaehlt"] = METHODEN["wertung"] != "kein FP"
METHODEN

In [ ]:
# calls: Aufrufe je Aktion, Messpunkt, Methode, Partei und Quelle, gebaut aus events
# events wächst nur an: die Aufrufe eines Messpunkts sind die ersten n_fp_calls Einträge (Messpunkt nach ist kumulativ)
from functools import lru_cache
from utils.origin_domain import origin_domain

@lru_cache(maxsize=None)
def domain_von(host):
    # Quelle als eTLD+1 wie im Crawler, 1P-Aufrufe haben keinen Host
    return origin_domain(host) if host else ""

call_zeilen = []
for aktion in AKTIONEN:
    for website, zeile in letzte[aktion].items():
        if art(zeile) != "gemessen":
            continue
        for punkt, key in PUNKTE.items():
            stats = zeile[key]
            if not stats:
                continue
            zaehler = Counter()
            for event in zeile["events"][:stats["n_fp_calls"]]:
                # third_party: True, False oder "Keine py_url" (Zuordnung im Browser nicht möglich)
                partei = {True: "3P", False: "1P"}.get(event["third_party"], "unbekannt")
                zaehler[(f"{event['api']}.{event['method']}", partei, domain_von(event["script_url"]))] += 1
            # Gegenprobe je Messpunkt: 3P-Aufrufe je api aus den ersten n_fp_calls events == fp_calls_per_api_3p des Crawlers
            api_3p = Counter()
            for (methode, partei, _), anzahl in zaehler.items():
                if partei == "3P":
                    api_3p[methode.rpartition(".")[0]] += anzahl
            assert dict(api_3p) == stats["fp_calls_per_api_3p"], (website, aktion, punkt)
            for (methode, partei, quelle_domain), anzahl in zaehler.items():
                call_zeilen.append({"aktion": aktion, "website": website, "punkt": punkt, "methode": methode,
                                    "partei": partei, "quelle": quelle_domain, "anzahl": anzahl})

calls = pd.DataFrame(call_zeilen).merge(METHODEN, on="methode", how="left", validate="many_to_one")
# Jede Methode braucht eine Wertung, sonst fehlt sie in METHODEN
assert calls["kategorie"].notna().all(), sorted(calls.loc[calls["kategorie"].isna(), "methode"].unique())
print(len(calls), "Zeilen")
calls.groupby(["aktion", "punkt", "partei"])["anzahl"].sum().unstack().loc[AKTIONEN]

# Analyse

Messpunkt vor = vor der Entscheidung (10 s nach dem Laden). Messpunkt nach = 30 s nach dem Laden, in allen drei Läufen gleich spät. Aufrufe nach der Entscheidung = nach - vor. Gezählt werden nur 3P-Aufrufe von Methoden, die als Fingerprinting zählen.

- Wirksamkeit der Aktion: Wo hat Zustimmen bzw. Ablehnen funktioniert? (Vergleichsgruppe)
- Vor der Entscheidung: Websites mit Banner und Fingerprinting vor jeder Entscheidung (H3, Teil 1)
- Nach der Entscheidung: Aufrufe nach keiner Aktion, Zustimmung und Ablehnung (H3, Teil 2)
- Website für Website, Techniken und Anbieter nach Ablehnung
- Cookies und Fingerprinting nach Ablehnung im Vergleich
- Reihenfolge der Besuche

In [ ]:
# Stil und Export für die Thesis: SVG mit Text als Text, LaTeX setzt die Beschriftung in der Schrift der Arbeit (vgl. analysis/abbildungen.md)
# In LaTeX: {\footnotesize \includesvg[width=\textwidth]{figures/ff3/<name>}}, übersetzen mit -shell-escape
import matplotlib.pyplot as plt

plt.rcParams.update({
    "svg.fonttype": "none", # Text bleibt Text, sonst werden Buchstaben zu Pfaden
    "axes.unicode_minus": False, # normales Minus, pdfLaTeX kennt das Unicode-Minus nicht
    "font.size": 9, "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.spines.top": False, "axes.spines.right": False,
})
BREITE = 135 / 25.4 # \textwidth der Thesis = 135 mm, in Zoll
ABBILDUNGEN = ROOT.parent / "Fingerprinting-Bachelor" / "figures" / "ff3"
# Keine LaTeX-Sonderzeichen (% _ & #) in Beschriftungen, deshalb "in Prozent" statt "%"

def speichern(fig, name):
    ABBILDUNGEN.mkdir(parents=True, exist_ok=True)
    fig.savefig(ABBILDUNGEN / f"{name}.svg")

# Eine Farbe je Aktion in allen Abbildungen
FARBEN = {"keine_aktion": "0.55", "zustimmen": "C2", "ablehnen": "C3"}

from matplotlib.ticker import FuncFormatter, NullFormatter

def zahlenachse(achse):
    # Log-Achse mit ganzen Zahlen (1, 10, 100, 1.000) statt 10 hoch x: Hochzahlen aus der SVG setzt LaTeX sonst als normale Ziffern
    achse.set_major_formatter(FuncFormatter(lambda wert, _: f"{wert:,.0f}".replace(",", ".")))
    achse.set_minor_formatter(NullFormatter())

In [ ]:
# Grundlage: Panel, nur 3P, nur Methoden, die zählen. Je Website, Aktion und Methode die Aufrufe vor und nach der Entscheidung
# Websites ohne Aufruf zählen mit 0. danach = nach - vor (Messpunkt nach ist kumulativ)
auswahl = calls[calls["website"].isin(panel_menge) & (calls["partei"] == "3P") & calls["zaehlt"]]
je_methode = (auswahl.pivot_table(index=["aktion", "website", "methode", "wertung"], columns="punkt", values="anzahl", aggfunc="sum", fill_value=0)
              .reset_index())
je_methode["danach"] = je_methode["nach"] - je_methode["vor"]

# Je Website und Aktion: Aufrufe vor / nach der Entscheidung, gibt es starke Aufrufe vor / nach der Entscheidung
je_methode["stark_vor"] = (je_methode["wertung"] == "stark") & (je_methode["vor"] > 0)
je_methode["stark_danach"] = (je_methode["wertung"] == "stark") & (je_methode["danach"] > 0)
summen = je_methode.groupby(["aktion", "website"]).agg(vor=("vor", "sum"), danach=("danach", "sum"),
                                                       stark_vor=("stark_vor", "any"), stark_danach=("stark_danach", "any"))
basis = pd.MultiIndex.from_product([AKTIONEN, panel], names=["aktion", "website"])
je_website = summen.reindex(basis).fillna({"vor": 0, "danach": 0, "stark_vor": False, "stark_danach": False}).astype({"stark_vor": bool, "stark_danach": bool})

# Tabellen Website x Aktion
vor = je_website["vor"].unstack("aktion")[AKTIONEN]
danach = je_website["danach"].unstack("aktion")[AKTIONEN]
stark_vor = je_website["stark_vor"].unstack("aktion")[AKTIONEN]
stark_danach = je_website["stark_danach"].unstack("aktion")[AKTIONEN]
print(len(panel), "Websites im Panel")
pd.DataFrame({"Median vor": vor.median(), "Median danach": danach.median(), "danach > 0 (in Prozent)": (danach > 0).mean() * 100}).round(1)

**Grundlage**
- Vor der Entscheidung sind alle drei Läufe gleich (Median 96 / 96 / 95 Aufrufe): passt, der Zustand vor dem Banner hängt nicht von der späteren Aktion ab
- Nach der Entscheidung im ganzen Panel: Median 0 (keine Aktion), 4 (Zustimmung), 0 (Ablehnung). Viele Websites haben keinen Banner, deshalb die Vergleichsgruppe

In [ ]:
# Wirksamkeit der Aktion: Wo hat Zustimmen bzw. Ablehnen funktioniert? Wie in der Thesis: Klick in beiden Läufen,
# und wo die Entscheidung lesbar ist (TCF, OneTrust, Cookiebot), muss sie zur Aktion passen. Ohne Aktion darf die Seite nichts selbst entscheiden
p = snap[snap["im_panel"]].set_index(["aktion", "website"])
klick = p["consent_klick"].unstack("aktion")
geprueft = p["geprueft"].unstack("aktion")
banner = klick["zustimmen"] | klick["ablehnen"] # Button zum Zustimmen oder Ablehnen gefunden
zustimmen_wirksam = klick["zustimmen"] & (geprueft["zustimmen"] != False)
ablehnen_wirksam = klick["ablehnen"] & (geprueft["ablehnen"] != False)
ruhig = geprueft["keine_aktion"] == True
vergleich = klick.index[zustimmen_wirksam & ablehnen_wirksam & ruhig].tolist() # Vergleichsgruppe für "nach der Entscheidung"
trichter = pd.Series({"Panel": len(klick), "Banner erkannt": banner.sum(), "Zustimmen wirksam": zustimmen_wirksam.sum(),
                      "Ablehnen wirksam": ablehnen_wirksam.sum(), "beides wirksam und ohne Aktion keine Entscheidung": len(vergleich)})
print(trichter.to_string())
print("Prüfung der Entscheidung (True = passt, False = falsch, None = nicht lesbar):")
pd.DataFrame({aktion: geprueft[aktion].map(repr).value_counts() for aktion in AKTIONEN}).fillna(0).astype(int)

**Wirksamkeit der Aktion**
- Banner erkannt auf 269 von 575 Websites, Zustimmen wirksam auf 257, Ablehnen nur auf 157: viele Banner bieten auf der ersten Ebene keinen Button zum Ablehnen
- Vergleichsgruppe: 149 Websites, auf denen beides funktioniert hat und die Seite ohne Aktion nichts selbst entschieden hat
- Lesbar ist die Entscheidung nur auf einem Teil (116 bzw. 72 passend), sonst zählt der Klick wie in der Thesis beschrieben. Falsche Entscheidungen nur je 4

In [ ]:
# Vor der Entscheidung (H3, Teil 1): Anteil der Websites mit 3P-Aufruf am ersten Messpunkt, getrennt nach Banner erkannt / nicht erkannt
# Vor der Entscheidung ist der Zustand in allen Läufen gleich, genommen wird der Lauf ohne Aktion. Fehlerbalken = 95-%-Konfidenzintervall nach Wilson
from statsmodels.stats.proportion import proportion_confint

gruppen = {"mit Banner": banner.index[banner], "ohne erkannten Banner": banner.index[~banner]}
zeilen_vor = []
for gruppe, liste in gruppen.items():
    for merkmal, werte in (("mind. ein 3P-Aufruf", vor.loc[liste, "keine_aktion"] > 0), ("mind. ein starker Aufruf", stark_vor.loc[liste, "keine_aktion"])):
        unten, oben = proportion_confint(werte.sum(), len(werte), method="wilson")
        zeilen_vor.append({"gruppe": gruppe, "merkmal": merkmal, "websites": len(werte), "anteil": werte.mean() * 100,
                           "unten": unten * 100, "oben": oben * 100})
vor_entscheidung = pd.DataFrame(zeilen_vor)

fig, ax = plt.subplots(figsize=(BREITE, BREITE * 0.42), layout="constrained")
for i, (gruppe, farbe) in enumerate(zip(gruppen, ("C0", "0.6"))):
    teil = vor_entscheidung[vor_entscheidung["gruppe"] == gruppe]
    x = [j + (i - 0.5) * 0.38 for j in range(len(teil))]
    ax.bar(x, teil["anteil"], width=0.38, color=farbe, label=f"{gruppe} (n = {teil['websites'].iloc[0]})",
           yerr=[teil["anteil"] - teil["unten"], teil["oben"] - teil["anteil"]], capsize=3)
ax.set_xticks(range(2), ["mind. ein 3P-Aufruf", "mind. ein starker Aufruf"])
ax.set_ylabel("Anteil der Websites (in Prozent)")
ax.set_ylim(0, 100)
ax.legend(frameon=False)
speichern(fig, "vor_entscheidung")
plt.show()
vor_entscheidung.round(1)

**Vor der Entscheidung (H3, Teil 1)**
- Websites mit Banner: 94,4 % rufen schon vor jeder Entscheidung FP-Schnittstellen aus 3P-Skripten auf (95-%-KI 91,0 bis 96,6 %), 27,9 % sogar starke (22,9 bis 33,5 %)
- Websites ohne erkannten Banner: 71,2 % bzw. 21,6 %
- H3, Teil 1 wird deutlich gestützt: Fingerprinting vor der Einwilligung ist die Regel, nicht die Ausnahme

In [ ]:
# Nach der Entscheidung (H3, Teil 2): Aufrufe zwischen erstem und zweitem Messpunkt je Aktion, nur Vergleichsgruppe
# Box = mittlere 50 %, Strich = Median. symlog zeigt auch Websites ohne Aufruf (0)
d = danach.loc[vergleich]
fig, ax = plt.subplots(figsize=(BREITE, BREITE * 0.45), layout="constrained")
ax.boxplot([d[aktion] for aktion in AKTIONEN], tick_labels=[KURZ[aktion] for aktion in AKTIONEN],
           showfliers=False, widths=0.5, medianprops={"color": "black"})
ax.set_yscale("symlog", linthresh=1)
zahlenachse(ax.yaxis)
ax.set_ylabel("3P-Aufrufe nach der Entscheidung")
speichern(fig, "nach_entscheidung")
plt.show()
pd.DataFrame({"Median": d.median(), "Mittel": d.mean(), "danach > 0 (in Prozent)": (d > 0).mean() * 100,
              "starker Aufruf danach (in Prozent)": stark_danach.loc[vergleich].mean() * 100}).round(1)

**Nach der Entscheidung (H3, Teil 2)**
- Vergleichsgruppe (149): Median der Aufrufe nach der Entscheidung 0 (keine Aktion), 14 (Ablehnung), 201 (Zustimmung)
- Auf 83,2 % der Websites gibt es nach einer Ablehnung weitere 3P-Aufrufe, ohne Aktion auf 26,2 %, nach Zustimmung auf 91,3 %
- Starke Aufrufe nach der Entscheidung: 2,7 % / 30,9 % / 16,1 %
- Die Ablehnung wirkt also teilweise (deutlich weniger als nach Zustimmung), aber nicht vollständig

In [ ]:
# Website für Website: Aufrufe nach der Entscheidung, Ablehnung gegen keine Aktion und gegen Zustimmung (Vergleichsgruppe)
# Punkte über der Diagonale = mehr Aufrufe nach Ablehnung. +1, damit Websites ohne Aufruf auf der Log-Achse erscheinen
fig, achsen = plt.subplots(1, 2, figsize=(BREITE, BREITE * 0.5), layout="constrained", sharex=True, sharey=True)
for ax, gegen in zip(achsen, ("keine_aktion", "zustimmen")):
    x, y = d[gegen] + 1, d["ablehnen"] + 1
    ax.scatter(x, y, s=8, alpha=0.6, color=FARBEN["ablehnen"], linewidths=0)
    grenze = max(x.max(), y.max()) * 1.5
    ax.plot([1, grenze], [1, grenze], color="black", linewidth=0.8)
    ax.set_xscale("log")
    ax.set_yscale("log")
    zahlenachse(ax.xaxis)
    zahlenachse(ax.yaxis)
    ax.set_title(f"gegen {KURZ[gegen]}")
    ax.text(0.03, 0.97, f"{(y > x).sum()} mehr\n{(y < x).sum()} weniger", transform=ax.transAxes, va="top")
    ax.set_xlabel(f"Aufrufe + 1, {KURZ[gegen]}")
achsen[0].set_ylabel("Aufrufe + 1, Ablehnung")
speichern(fig, "website_vergleich")
plt.show()

**Website für Website**
- Ablehnung gegen keine Aktion: 118 Websites mit mehr Aufrufen nach der Ablehnung, nur 2 mit weniger
- Ablehnung gegen Zustimmung: 120 mit weniger, 6 mit mehr
- Die Ablehnung liegt fast immer zwischen den beiden: weniger als Zustimmung, mehr als gar nichts tun

In [ ]:
# Techniken nach der Entscheidung: Anteil der Websites, auf denen eine Methode nach der Entscheidung neue 3P-Aufrufe hat (Vergleichsgruppe)
# Nur Methoden, die bei einer Aktion auf mind. 2 % der Websites vorkommen. Zeilen: stark oben, dann nach Anteil bei Ablehnung
neu = je_methode[je_methode["website"].isin(vergleich) & (je_methode["danach"] > 0)]
heat = (neu.groupby(["methode", "aktion"])["website"].nunique().unstack(fill_value=0)
        .reindex(columns=AKTIONEN, fill_value=0) / len(vergleich) * 100)
heat = heat[heat.max(axis=1) >= 2]
info = METHODEN.set_index("methode").loc[heat.index]
heat = heat.loc[info.assign(ablehnen=heat["ablehnen"]).sort_values(["wertung", "ablehnen"], ascending=[False, False]).index]

fig, ax = plt.subplots(figsize=(BREITE, 0.2 * len(heat) + 0.8), layout="constrained")
ax.pcolormesh(heat.values, cmap="Blues", vmin=0, vmax=max(heat.values.max(), 1))
for zeile, werte in enumerate(heat.values):
    for spalte, wert in enumerate(werte):
        ax.text(spalte + 0.5, zeile + 0.5, f"{wert:.0f}", ha="center", va="center", fontsize=7,
                color="white" if wert > 0.6 * heat.values.max() else "black")
ax.set_xticks([i + 0.5 for i in range(len(AKTIONEN))], [KURZ[aktion] for aktion in AKTIONEN])
ax.set_yticks([i + 0.5 for i in range(len(heat))], [f"{m.rpartition('.')[2]} ({info.loc[m, 'kategorie']})" for m in heat.index])
ax.invert_yaxis()
ax.xaxis.tick_top()
ax.tick_params(length=0)
for seite in ax.spines.values():
    seite.set_visible(False)
ax.axhline((info.loc[heat.index, "wertung"] == "stark").sum(), color="black", linewidth=0.8) # Trennlinie stark / schwach
speichern(fig, "techniken_nach_entscheidung")
plt.show()

**Techniken nach der Entscheidung**
- Nach der Ablehnung lesen 3P-Skripte auf 73 % der Websites den User-Agent neu aus, auf 48 % die Bildschirmgröße, auf 19 % die Client Hints
- Stark: WebGL getParameter nach Ablehnung auf 16 % der Websites (keine Aktion 3 %, Zustimmung 31 %), Canvas und Audio je 3 %

In [ ]:
# Anbieter nach der Entscheidung: Auf wie vielen Websites der Vergleichsgruppe ruft eine Quelle nach der Entscheidung noch auf?
# neu_nach_ablehnung: Quelle war vor der Entscheidung nicht aktiv und ruft erst nach der Ablehnung auf
je_quelle = (auswahl[auswahl["website"].isin(vergleich)]
             .pivot_table(index=["aktion", "website", "quelle"], columns="punkt", values="anzahl", aggfunc="sum", fill_value=0).reset_index())
aktiv = je_quelle[je_quelle["nach"] > je_quelle["vor"]]
anbieter = aktiv.groupby(["quelle", "aktion"])["website"].nunique().unstack(fill_value=0).reindex(columns=AKTIONEN, fill_value=0)
anbieter["neu_nach_ablehnung"] = aktiv[(aktiv["aktion"] == "ablehnen") & (aktiv["vor"] == 0)].groupby("quelle")["website"].nunique()
anbieter.fillna({"neu_nach_ablehnung": 0}).astype(int).sort_values("ablehnen", ascending=False).head(12)

**Anbieter nach der Entscheidung**
- Häufigste Quellen nach Ablehnung: cookielaw.org (39, das Consent-Tool von OneTrust), googletagmanager.com (32), googlesyndication.com (21), doubleclick.net (20)
- googlesyndication.com ruft auf 16 Websites erst nach der Ablehnung auf, war vorher nicht aktiv: Werbung lädt trotz Ablehnung nach (bei TCF ggf. über berechtigtes Interesse oder nicht personalisierte Werbung)
- Unter den Quellen sind mehrere Consent-Tools (cookielaw.org, cookiebot.com, osano.com, trustarc.com): sie reagieren auf den Klick, siehe Gegenprobe

In [ ]:
# Gegenprobe ohne Consent-Tools: Consent-Tools lesen beim Bedienen des Banners selbst Browser-Merkmale aus (z. B. cookielaw.org von OneTrust)
# Das ist kein Tracking. Wie viele Aufrufe nach der Entscheidung bleiben ohne ihre Quellen? (Liste bekannter Consent-Tools, nicht vollständig)
CONSENT_TOOLS = {"cookielaw.org", "onetrust.com", "cookiepro.com", "cookiebot.com", "usercentrics.eu", "osano.com", "trustarc.com", "truste.com",
                 "privacy-center.org", "didomi.io", "consensu.org", "privacy-mgmt.com", "cookieyes.com", "cookie-script.com", "iubenda.com",
                 "consentmanager.net", "cookiefirst.com", "termly.io"}
ohne_tools = je_quelle[~je_quelle["quelle"].isin(CONSENT_TOOLS)]
d_ohne = ((ohne_tools["nach"] - ohne_tools["vor"]).groupby([ohne_tools["aktion"], ohne_tools["website"]]).sum()
          .reindex(pd.MultiIndex.from_product([AKTIONEN, vergleich], names=["aktion", "website"]), fill_value=0).unstack("aktion")[AKTIONEN])
from scipy.stats import wilcoxon
w = wilcoxon(d_ohne["keine_aktion"], d_ohne["ablehnen"], method="approx")
print(f"Ablehnung gegen keine Aktion ohne Consent-Tools: p = {w.pvalue:.4g}, r = {abs(w.zstatistic) / len(d_ohne) ** 0.5:.2f}")
pd.DataFrame({"Median": d_ohne.median(), "danach > 0 (in Prozent)": (d_ohne > 0).mean() * 100,
              "Anteil Consent-Tools an den Aufrufen danach (in Prozent)": (1 - d_ohne.sum() / d.sum()) * 100}).round(1)

**Gegenprobe ohne Consent-Tools**
- Consent-Tools machen 38,8 % der Aufrufe nach einer Ablehnung aus
- Ohne sie bleiben auf 63,8 % der Websites Aufrufe nach der Ablehnung (Median 6), ohne Aktion auf 24,8 % (Median 0)
- Der Unterschied bleibt deutlich (p < 0,001, r = 0,64): auch ohne Consent-Tools geht Fingerprinting nach der Ablehnung weiter

In [ ]:
# Cookies und Fingerprinting nach der Entscheidung (Vergleichsgruppe): neue 3P-Cookie-Domains bzw. neue FP-Quellen nach der Entscheidung
# Ein Verstoß mit Cookies bleibt im Browser sichtbar, einer mit Fingerprinting nicht. Wie oft kommt beides nach einer Ablehnung vor?
v = snap[snap["website"].isin(vergleich) & (snap["art"] == "gemessen")].set_index(["aktion", "website"])
neue_cookies = (v["cookie_domains_3p_nach"].combine(v["cookie_domains_3p_vor"], lambda nachher, vorher: len(nachher - vorher) > 0)).unstack("aktion")[AKTIONEN]
neue_quellen = (aktiv[aktiv["vor"] == 0].groupby(["aktion", "website"]).size() > 0).reindex(
    pd.MultiIndex.from_product([AKTIONEN, vergleich], names=["aktion", "website"]), fill_value=False).unstack("aktion")[AKTIONEN]
vergleich_tab = pd.DataFrame({"neue 3P-Cookies": neue_cookies.mean() * 100, "neue FP-Quellen": neue_quellen.mean() * 100})

fig, ax = plt.subplots(figsize=(BREITE, BREITE * 0.5), layout="constrained")
x = range(len(AKTIONEN))
ax.bar([i - 0.2 for i in x], vergleich_tab["neue 3P-Cookies"], width=0.4, color="0.45", label="neue 3P-Cookie-Domains")
ax.bar([i + 0.2 for i in x], vergleich_tab["neue FP-Quellen"], width=0.4, color="C3", label="neue Fingerprinting-Quellen")
ax.set_xticks(list(x), [KURZ[aktion] for aktion in AKTIONEN])
ax.set_ylabel("Anteil der Websites (in Prozent)")
ax.set_ylim(0, 100)
ax.legend(frameon=False, loc="upper left")
speichern(fig, "cookies_und_fingerprinting")
plt.show()
print("Ablehnung: beides", (neue_cookies["ablehnen"] & neue_quellen["ablehnen"]).sum(), "| nur FP", (~neue_cookies["ablehnen"] & neue_quellen["ablehnen"]).sum(),
      "| nur Cookies", (neue_cookies["ablehnen"] & ~neue_quellen["ablehnen"]).sum(), "von", len(vergleich), "Websites")
vergleich_tab.round(1)

**Cookies und Fingerprinting nach der Ablehnung**
- Neue 3P-Cookie-Domains nach Ablehnung auf 6,7 % der Websites, neue FP-Quellen auf 18,8 %
- 19 Websites haben nach der Ablehnung nur neue FP-Quellen, 9 beides, 1 nur neue Cookies
- Fingerprinting nach Ablehnung ist fast dreimal so häufig wie neue Cookies. Von 29 Websites mit neuen Cookies oder FP-Quellen hätten 19 im Browser keinen Beleg

In [ ]:
# Reihenfolge der Besuche: wie oft jede Aktion an 1., 2., 3. Stelle stand, und Median der Aufrufe nach der Entscheidung je Stelle
pos = snap[snap["im_panel"]].set_index(["aktion", "website"])["position"]
tabelle = pd.DataFrame({"position": pos, "danach": je_website["danach"]}).reset_index()
print(pd.crosstab(tabelle["aktion"], tabelle["position"]).loc[AKTIONEN])
pd.pivot_table(tabelle, index="aktion", columns="position", values="danach", aggfunc="median").loc[AKTIONEN]

**Reihenfolge**
- Jede Aktion stand ungefähr gleich oft an 1., 2. und 3. Stelle (171 bis 216 Websites)
- Die Mediane je Stelle bleiben ähnlich, kein Reihenfolgeeffekt zu sehen
- Hinweis: Der Crawler wechselt die Reihenfolge reihum wie bei FF2, die Thesis schreibt bei FF3 zufällig. Text angleichen

## Statistik

- Aufrufe nach der Entscheidung: Friedman über die drei Aktionen, Wilcoxon paarweise mit Effektstärke r, Holm-Korrektur
- Starker Aufruf nach der Entscheidung: Cochran-Q, McNemar paarweise
- Vor der Entscheidung: Anteile mit Wilson-Intervall (siehe Abbildung)

In [ ]:
# Aufrufe nach der Entscheidung: Unterschied zwischen den Aktionen? Dieselben Websites in allen drei Läufen, deshalb gepaarte Rangtests
# Effektstärke r = |z| / Wurzel(n), grob: 0,1 klein, 0,3 mittel, 0,5 groß
from scipy.stats import friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests

f = friedmanchisquare(*[d[aktion] for aktion in AKTIONEN])
print(f"Friedman = {f.statistic:.1f}, p = {f.pvalue:.4g}")
PAARE = [("keine_aktion", "ablehnen"), ("ablehnen", "zustimmen"), ("keine_aktion", "zustimmen")]
zeilen_wilcoxon = []
for a, b in PAARE:
    w = wilcoxon(d[a], d[b], method="approx")
    zeilen_wilcoxon.append({"vergleich": f"{KURZ[a]} -> {KURZ[b]}", "median_a": d[a].median(), "median_b": d[b].median(),
                            "p": w.pvalue, "r": abs(w.zstatistic) / len(d) ** 0.5})
test_aufrufe = pd.DataFrame(zeilen_wilcoxon)
test_aufrufe["p_holm"] = multipletests(test_aufrufe["p"], method="holm")[1]
test_aufrufe.round(4)

**Test: Aufrufe nach der Entscheidung**
- Friedman p < 0,001: die Aktionen unterscheiden sich deutlich
- Keine Aktion -> Ablehnung: 0 -> 14, r = 0,74 (großer Effekt), Ablehnung -> Zustimmung: 14 -> 201, r = 0,74, alle p < 0,001 nach Holm
- H3, Teil 2 wird gestützt: nach einer Ablehnung gibt es deutlich mehr 3P-Aufrufe als ohne jede Aktion

In [ ]:
# Starker Aufruf nach der Entscheidung: Cochran-Q über alle drei Aktionen, McNemar exakt paarweise mit Holm-Korrektur (Vergleichsgruppe)
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar

s = stark_danach.loc[vergleich]
q = cochrans_q(s.astype(int))
print(f"Cochran-Q = {q.statistic:.1f}, p = {q.pvalue:.4f}")
zeilen_mcnemar = []
for a, b in PAARE:
    # Vierfeldertafel: Zeilen = Aktion a stark ja/nein, Spalten = Aktion b stark ja/nein
    tafel = pd.crosstab(s[a], s[b]).reindex(index=[False, True], columns=[False, True], fill_value=0)
    zeilen_mcnemar.append({"vergleich": f"{KURZ[a]} -> {KURZ[b]}", "nur_b_stark": tafel.loc[False, True],
                           "nur_a_stark": tafel.loc[True, False], "p": mcnemar(tafel, exact=True).pvalue})
test_stark = pd.DataFrame(zeilen_mcnemar)
test_stark["p_holm"] = multipletests(test_stark["p"], method="holm")[1]
test_stark.round(4)

**Test: starker Aufruf nach der Entscheidung**
- Keine Aktion -> Ablehnung: auf 21 Websites kommt ein starker Aufruf erst nach der Ablehnung dazu, bei 1 umgekehrt
- Ablehnung -> Zustimmung: 23 zu 1, alle p < 0,001 nach Holm

# Nebenfakten

- Auswahl des Panels: Verteilung auf die Rangbereiche
- Robustheit: nur FF1-Stichprobe, nur geprüfte Entscheidungen, nur TCF-Websites
- Klick-Kontrolle: Wie hat der Crawler den Button gefunden, welche Buttons waren es? (für die Sichtung von Hand)

In [ ]:
# Auswahl des Panels prüfen: Sind die Rangbereiche im Panel noch gleich stark vertreten?
# Rang aus der Tranco-Liste, Grenzen der Rangbereiche aus der Konfiguration (wie beim Ziehen der Stichprobe, utils/tranco_worker.py)
import csv
from itertools import islice

GRENZEN = configs["zustimmen"][-1]["rank_bands"]
with open(configs["zustimmen"][-1]["path_to_webpages"], encoding="utf-8") as datei:
    rang = {domain: int(platz) for platz, domain in islice(csv.reader(datei), GRENZEN[-1])}

def rangbereich(website):
    platz = rang[website.removeprefix("https://")]
    return next(bereich for bereich, grenze in enumerate(GRENZEN, 1) if platz <= grenze)

for name, liste in {"Stichprobe": websites, "vollständig": vollstaendig, "Panel": panel}.items():
    zaehler = Counter(rangbereich(website) for website in liste)
    print(f"{name:12s} n={len(liste):3d}  je Rangbereich:", {bereich: zaehler[bereich] for bereich in range(1, len(GRENZEN) + 1)})

**Rangbereiche**
- Panel 575 von 1000 Websites, der Bereich der Top 1000 ist etwas schwächer vertreten (124 gegen 165 im Bereich 4)

In [ ]:
# Robustheit: Hält das Ergebnis (Aufrufe nach Ablehnung gegen keine Aktion) auch für Teilgruppen der Vergleichsgruppe?
# Geprüft: Entscheidung bei Zustimmen und Ablehnen lesbar und passend (True). TCF: Website nutzt das Transparency and Consent Framework
ff1_websites = {json.loads(line).get("website") for line in open("results/Forschungsfrage 1/ff1_live.jsonl", encoding="utf-8")}
geprueft_beide = geprueft.index[(geprueft["zustimmen"] == True) & (geprueft["ablehnen"] == True)]
tcf_websites = p.loc["zustimmen"].index[p.loc["zustimmen"]["tcf_status"].notna()]
teilgruppen = {"Vergleichsgruppe": d.index, "nur FF1-Stichprobe": d.index.intersection(list(ff1_websites)),
               "nur geprüfte Entscheidung": d.index.intersection(geprueft_beide), "nur TCF-Websites": d.index.intersection(tcf_websites)}
for name, teilgruppe in teilgruppen.items():
    teil = d.loc[teilgruppe]
    w = wilcoxon(teil["keine_aktion"], teil["ablehnen"], method="approx") if len(teil) > 10 else None
    print(f"{name:26s} n={len(teil):3d}  danach > 0 bei Ablehnung {(teil['ablehnen'] > 0).mean():.0%}  "
          f"Median keine Aktion {teil['keine_aktion'].median():5.1f} / Ablehnung {teil['ablehnen'].median():5.1f} / Zustimmung {teil['zustimmen'].median():5.1f}  "
          + (f"p={w.pvalue:.4f} r={abs(w.zstatistic) / len(teil) ** 0.5:.2f}" if w else "zu wenige Websites für den Test"))

**Robustheit**
- FF1-Stichprobe (79): gleiches Bild, 82 % mit Aufrufen nach Ablehnung
- Nur geprüfte Entscheidung (69): sogar 96 %, Median nach Ablehnung 38
- Nur TCF-Websites (32): 100 %, Median nach Ablehnung 337 (Zustimmung 1296). TCF-Seiten, oft Nachrichten, fingerprinten nach einer Ablehnung besonders viel
- Das Ergebnis hängt also nicht an der Art der Prüfung, eher wird es mit strengerer Prüfung deutlicher

In [ ]:
# Klick-Kontrolle: Über welche Stufe hat der Crawler den Button gefunden (1 allgemeine CSS-Selektoren, 2 bekannte CMPs, 3 Button-Text)?
# Darunter die häufigsten Button-Texte je Aktion, zum Prüfen von Hand (falsche Buttons wie "Einstellungen" fallen hier auf)
geklickt = snap[snap["im_panel"] & snap["consent_klick"]]
print(pd.crosstab(geklickt["aktion"], geklickt["klick_stufe"]))
for aktion in ("zustimmen", "ablehnen"):
    print(f"\n{KURZ[aktion]}:", Counter(geklickt.loc[geklickt["aktion"] == aktion, "button"].fillna("").str.lower()).most_common(10))

**Klick-Kontrolle**
- Gut die Hälfte (Zustimmung) bzw. zwei Drittel (Ablehnung) der Klicks kommen über allgemeine CSS-Selektoren, der Rest über bekannte CMPs oder den Button-Text
- Die häufigsten Buttons passen: "alle akzeptieren", "accept all" bzw. "reject all", "alle ablehnen", "nicht wesentliche cookies ablehnen"
- Von Hand prüfen: Buttons wie "ok" oder "consent" (Zustimmung) und die Screenshots dazu